In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
import os
import glob
import sys
import warnings

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from tqdm import tqdm_notebook
from cogwheel import plotting, gw_plotting, utils, gw_utils
from cogwheel.likelihood.marginalization.coherent_score_lensing import bootstrap_bayes_factors

sys.path.insert(0, '/home/abbye.williams/GWPE/code')
# from generate_injection_params import generate_injection_parameters
import helpers

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ["MPLCONFIGDIR"] = f"/tmp/{os.environ['USER']}_matplotlib_cache"
warnings.simplefilter("ignore", RuntimeWarning)

In [3]:
# load injection parameters
approximant = 'IMRPhenomXPHM'
save_fn = f'/home/abbye.williams/GWPE/data/injections/Halton/{approximant}/injection_params_{approximant}.pkl'
injection_params = pd.read_pickle(save_fn)

# get the SNR and compute the Bayes factor
resdir = f'/home/abbye.williams/GWPE/data/injections/Halton/{approximant}/IntrinsicLVCPrior'
seed = 12
nevents = 100
idx_existing = []
idx_to_run = []
n_resamples = 1000
h_h = {}
lnBayes = {}
# parameters to collect
params = {}
for idx in range(nevents):
    # load this sample
    eventname = f'GW{seed}_{idx}'
    eventdir = os.path.join(resdir, eventname)
    try:
        event_data, samples, samples_dir = helpers.load_event_data_and_posterior_samples(eventdir, eventname)
        # add cos iota
        samples['cosiota'] = np.cos(samples['iota'])
        idx_existing.append(idx)
    except FileNotFoundError:
        print(f"no samples for idx {idx}. continuing")
        idx_to_run.append(idx)
        continue

    # add derived quantities to the injection dictionary
    injection_dict = event_data.get_init_dict()['injection']
    injected_par_dict = injection_dict['par_dic']
    helpers.add_derived_quantities_injection(injected_par_dict)
    # add cos iota
    injected_par_dict['cosiota'] = np.cos(injected_par_dict['iota'])

    # optimal SNR: inner product from the injection
    h_h[idx] = np.sum(injection_dict['h_h'])

    # Bayes factor
    lnBayes_fn = os.path.join(samples_dir, 'lnBayes.npy')
    try:
        lnBayes[idx] = np.load(lnBayes_fn)
    except FileNotFoundError:
        bayes_factors = bootstrap_bayes_factors(samples, n_resamples)
        # log these
        lnBayes_factors = np.log(bayes_factors)
        # get the median and standard deviation
        lnBayes[idx] = (np.median(lnBayes_factors), np.std(lnBayes_factors))
        np.save(lnBayes_fn, lnBayes[idx])

    # median from the posterior
    for par_id, injected_value in injected_par_dict.items():
        median_post = utils.quantile(samples[par_id], 0.5, weights=samples['weights'])
        try:
            params[par_id][idx] = (injected_value, median_post)
        except KeyError:    # if the dictionary doesn't exist yet
            params[par_id] = {}
            params[par_id][idx] = (injected_value, median_post)

no samples for idx 0. continuing
loaded posterior samples for GW12_1 run 0
loaded posterior samples for GW12_2 run 0
loaded posterior samples for GW12_3 run 0
loaded posterior samples for GW12_4 run 0
loaded posterior samples for GW12_5 run 0
loaded posterior samples for GW12_6 run 0
loaded posterior samples for GW12_7 run 0
loaded posterior samples for GW12_8 run 0
loaded posterior samples for GW12_9 run 0
loaded posterior samples for GW12_10 run 0
loaded posterior samples for GW12_11 run 0
loaded posterior samples for GW12_12 run 0
loaded posterior samples for GW12_13 run 0
loaded posterior samples for GW12_14 run 0
loaded posterior samples for GW12_15 run 0
loaded posterior samples for GW12_16 run 0
no samples for idx 17. continuing
loaded posterior samples for GW12_18 run 0
no samples for idx 19. continuing
loaded posterior samples for GW12_20 run 0
loaded posterior samples for GW12_21 run 0
loaded posterior samples for GW12_22 run 0
loaded posterior samples for GW12_23 run 0
loade

In [4]:
# helper function
def plot_pII(samples, lnB):
    fig, axs = plt.subplots(1, 3, figsize=(15,4), gridspec_kw=dict(wspace=0.2))
    for ax, lidx, label, c in zip(axs, [np.full_like(samples['lensed'], True), samples['lensed'], ~samples['lensed']],
                                ['all', 'Type II lensed', 'unlensed'],
                                ['navy', 'royalblue', 'seagreen']):
        hist, edges = np.histogram(samples['p_lensed'][lidx], bins=100,
                                    weights=samples['weights'][lidx])
        # we want the pdf at edges, not midpoints:
        pdf = np.array([hist[0], *(hist[1:] + hist[:-1]) / 2, hist[-1]])

        ax.plot(edges, pdf, c=c, alpha=0.9, lw=2)
        ax.axvline(0.5, c='k', alpha=0.5, ls='--', lw=1.)
        ax.set_xlim(0., 1.)
        ax.set_ylim(0., None)
        ax.set_xlabel(r'$p_\mathrm{II}$', fontsize=14)

        # vlines
        median, *span = helpers.get_pII_median_span(samples[lidx])
        for val in (median, *span):
            ax.plot([val] * 2, [0, np.interp(val, edges, pdf)], c=c, alpha=0.8, lw=0.5)
        idx = (edges > span[0]) & (edges < span[1])
        ax.fill_between(edges[idx], np.zeros_like(pdf[idx]), pdf[idx], color=c, alpha=0.1)
        val_err_str = '${}={}$' + plotting.latex_val_err(median, np.subtract(median, span))
        ax.set_title(f'{label}: 'r'$p_\mathrm{II}$' + val_err_str)

    lnB_str = 'NaN' if np.isnan(lnB[0]) else f'{lnB[0]:.3f}'r'$\pm$'f'{lnB[1]:.3f}'
    title_str = f'{eventname}: 'r'$\ln\mathcal B =$'+lnB_str
    fig.suptitle(title_str, y=1.01)
    return fig

In [5]:
for i in idx_existing:
    # load this sample
    eventname = f'GW{seed}_{i}'
    save_fn = f'../data/plots/Halton_pII/{eventname}.pdf'
    if os.path.exists(save_fn):
        print(f"plot already saved for {eventname}. continuing")
        continue
    eventdir = os.path.join(resdir, eventname)
    event_data, samples, samples_dir = helpers.load_event_data_and_posterior_samples(eventdir, eventname)
    # injected parameters
    # inj_par_dict = event_data.get_init_dict()['injection']['par_dic']
    # helpers.add_derived_quantities_injection(inj_par_dict)
    # sanity check with lensing
    # helpers.lensed_sanity_check(samples)
    fig = plot_pII(samples, lnB=lnBayes[i])
    fig.savefig(save_fn, bbox_inches='tight', pad_inches=0.2)
    plt.close(fig)

plot already saved for GW12_1. continuing
plot already saved for GW12_2. continuing
plot already saved for GW12_3. continuing
plot already saved for GW12_4. continuing
plot already saved for GW12_5. continuing
plot already saved for GW12_6. continuing
plot already saved for GW12_7. continuing
plot already saved for GW12_8. continuing
plot already saved for GW12_9. continuing
plot already saved for GW12_10. continuing
plot already saved for GW12_11. continuing
plot already saved for GW12_12. continuing
plot already saved for GW12_13. continuing
plot already saved for GW12_14. continuing
plot already saved for GW12_15. continuing
plot already saved for GW12_16. continuing
plot already saved for GW12_18. continuing
plot already saved for GW12_20. continuing
plot already saved for GW12_21. continuing
plot already saved for GW12_22. continuing
plot already saved for GW12_23. continuing
plot already saved for GW12_24. continuing
plot already saved for GW12_25. continuing
plot already saved f

In [6]:
# combine pdfs
pdf_paths = [
    f'../data/plots/Halton_pII/GW{seed}_{i}.pdf' for i in idx_existing
]
pdf_paths_str = ' '.join(pdf_paths)
bigpdf_path = f'../data/plots/Halton_pII/Halton_pII.pdf'
!gs -dBATCH -dNOPAUSE -q -sDEVICE=pdfwrite -dPDFSETTINGS=/prepress -sOutputFile={bigpdf_path} {pdf_paths_str}

In [7]:
# investigate the lnB = NaN cases
for idx in idx_existing:
    # load this sample
    eventname = f'GW{seed}_{idx}'
    eventdir = os.path.join(resdir, eventname)
    event_data, samples, samples_dir = helpers.load_event_data_and_posterior_samples(eventdir, eventname, verbose=False)
    # Bayes factor
    lnB_med, lnB_std = np.load(os.path.join(samples_dir, 'lnBayes.npy'))
    if np.isnan(lnB_med):
        median, *span = helpers.get_pII_median_span(samples)
        isone = (median == 1.)
        print(f"{eventname:<10}:\tlnB = NaN\t--> pII = {median} ({span[0]}, {span[1]}). {isone}")

GW12_1    :	lnB = NaN	--> pII = 1.0 (0.9999999999999998, 1.0000000000000002). True
GW12_2    :	lnB = NaN	--> pII = 1.0 (0.9999999999999998, 1.0000000000000002). True
GW12_20   :	lnB = NaN	--> pII = 1.0 (0.9999999999999998, 1.0000000000000002). True
GW12_22   :	lnB = NaN	--> pII = 1.0 (0.9999999999999998, 1.0000000000000002). True
GW12_23   :	lnB = NaN	--> pII = 1.0 (0.9999999999999998, 1.0000000000000002). True
GW12_44   :	lnB = NaN	--> pII = 1.0 (0.9999999999999998, 1.0000000000000002). True
GW12_45   :	lnB = NaN	--> pII = 1.0 (0.9999999999999998, 1.0000000000000002). True
GW12_47   :	lnB = NaN	--> pII = 1.0 (0.9999999999999998, 1.0000000000000002). True
GW12_49   :	lnB = NaN	--> pII = 1.0 (0.9999999999999998, 1.0000000000000002). True
GW12_56   :	lnB = NaN	--> pII = 1.0 (0.9999999999999998, 1.0000000000000002). True
GW12_58   :	lnB = NaN	--> pII = 1.0 (0.9999999999999998, 1.0000000000000002). True
GW12_61   :	lnB = NaN	--> pII = 1.0 (0.9999999999999998, 1.0000000000000002). True
GW12

### corner plots

In [11]:
corner_plot_kwargs = dict(tail_probability=1e-3, color_2d='royalblue',
    kwargs_1d=dict(color='royalblue', alpha=0.8), contour_kwargs=dict(alpha=0.8))
for i in idx_existing:
    # load this sample
    eventname = f'GW{seed}_{i}'
    save_fn = f'../data/plots/Halton_corner/{eventname}.pdf'
    if os.path.exists(save_fn):
        print(f"plot already exists for {eventname}. continuing")
        continue
    eventdir = os.path.join(resdir, eventname)
    event_data, samples, samples_dir = helpers.load_event_data_and_posterior_samples(eventdir, eventname)
    samples['cosiota'] = np.cos(samples['iota'])
    pars_to_plot = [
        'mchirp', 'q', 'cosiota', 'psi', 'phi_ref', 'chieff', 'd_luminosity', 'ra', 'dec', 'p_lensed'
    ]
    try:
        cp = gw_plotting.CornerPlot(samples, params=pars_to_plot, **corner_plot_kwargs)
        cp.plot()
    except ValueError as e:  # if there's an issue plotting p_lensed, probably
        plt.close(cp.fig)
        print(f"ValueError: {e}. removing p_lensed and attempting to plot again...")
        pars_to_plot.remove('p_lensed')
        cp = gw_plotting.CornerPlot(samples, params=pars_to_plot, **corner_plot_kwargs)
        cp.plot()
    # injected parameters
    inj_par_dic = event_data.get_init_dict()['injection']['par_dic']
    helpers.add_derived_quantities_injection(inj_par_dic)
    inj_par_dic['cosiota'] = np.cos(inj_par_dic['iota'])
    inj_pars_to_plot = {k : v for k, v in inj_par_dic.items() if k in pars_to_plot}
    cp.scatter_points(inj_pars_to_plot, colors=['#FF2020'], s=300,
                                        zorder=2, marker='+', adjust_lims=True)
    lnB = lnBayes[i]
    lnB_str = 'NaN' if np.isnan(lnB[0]) else f'{lnB[0]:.3f}'r'$\pm$'f'{lnB[1]:.3f}'
    title_str = f'{eventname}: 'r'$\ln\mathcal B =$'+lnB_str
    cp.fig.suptitle(title_str, y=0.97)
    cp.fig.savefig(save_fn, bbox_inches='tight', pad_inches=0.2)
    plt.close(cp.fig)

loaded posterior samples for GW12_1 run 0
ValueError: Too many bins for data range. Cannot create 46 finite-sized bins.. removing p_lensed and attempting to plot again...
loaded posterior samples for GW12_2 run 0
ValueError: Too many bins for data range. Cannot create 48 finite-sized bins.. removing p_lensed and attempting to plot again...
loaded posterior samples for GW12_3 run 0
loaded posterior samples for GW12_4 run 0
loaded posterior samples for GW12_5 run 0
loaded posterior samples for GW12_6 run 0
loaded posterior samples for GW12_7 run 0
loaded posterior samples for GW12_8 run 0
loaded posterior samples for GW12_9 run 0
loaded posterior samples for GW12_10 run 0
loaded posterior samples for GW12_11 run 0
loaded posterior samples for GW12_12 run 0
loaded posterior samples for GW12_13 run 0
ValueError: Too many bins for data range. Cannot create 45 finite-sized bins.. removing p_lensed and attempting to plot again...
loaded posterior samples for GW12_14 run 0
loaded posterior sam

In [12]:
# combine pdfs
pdf_paths = [
    f'../data/plots/Halton_corner/GW{seed}_{i}.pdf' for i in idx_existing
]
pdf_paths_str = ' '.join(pdf_paths)
bigpdf_path = f'../data/plots/Halton_corner/Halton_corner.pdf'
!gs -dBATCH -dNOPAUSE -q -sDEVICE=pdfwrite -dPDFSETTINGS=/prepress -sOutputFile={bigpdf_path} {pdf_paths_str}

#### corner plots with lensed and unlensed

In [ ]:
c1, c2 = 'royalblue', 'mediumseagreen'
plotstyles = [
        plotting.PlotStyle(color_2d=c1, contour_kwargs=dict(alpha=0.7), kwargs_1d=dict(color=c1, alpha=0.7), tail_probability=1e-3),
        plotting.PlotStyle(color_2d=c2, contour_kwargs=dict(alpha=0.7), kwargs_1d=dict(color=c2, alpha=0.7), tail_probability=1e-3)
]
for i in idx_existing:
    # load this sample
    eventname = f'GW{seed}_{i}'
    save_fn = f'../data/plots/Halton_corner/{eventname}_split.pdf'
    if os.path.exists(save_fn):
        print(f"plot already exists for {eventname}. continuing")
        continue
    eventdir = os.path.join(resdir, eventname)
    event_data, samples, samples_dir = helpers.load_event_data_and_posterior_samples(eventdir, eventname)
    samples['cosiota'] = np.cos(samples['iota'])
    lensed_idx = samples['lensed']
    pars_to_plot = [
        'mchirp', 'q', 'iota', 'psi', 'phi_ref', 'chieff', 'd_luminosity', 'ra', 'dec', 'p_lensed'
    ]
    try:
        cp = gw_plotting.MultiCornerPlot({'lensed' : samples[lensed_idx], 'unlensed' : samples[~lensed_idx]},
                                            params=pars_to_plot, plotstyles=plotstyles)
        cp.plot()
    except ValueError as e:  # if there's an issue plotting p_lensed, probably
        plt.close(cp.corner_plots[0].fig)
        print(f"ValueError: {e}. removing p_lensed and attempting to plot again...")
        pars_to_plot.remove('p_lensed')
        cp = gw_plotting.MultiCornerPlot({'lensed' : samples[lensed_idx], 'unlensed' : samples[~lensed_idx]},
                                            params=pars_to_plot, plotstyles=plotstyles)
        try:
            cp.plot()
        except Exception as e:
            plt.close(cp.corner_plots[0].fig)
            print(f"exception! {e} continuing to next event.")
            continue
    except Exception as e:
        plt.close(cp.corner_plots[0].fig)
        print(f"exception! {e} continuing to next event.")
        continue
    # injected parameters
    inj_par_dic = event_data.get_init_dict()['injection']['par_dic']
    helpers.add_derived_quantities_injection(inj_par_dic)
    inj_par_dic['cosiota'] = np.cos(inj_par_dic['iota'])
    inj_pars_to_plot = {k : v for k, v in inj_par_dic.items() if k in pars_to_plot}
    cp.scatter_points(inj_pars_to_plot, colors=['#FF2020'], s=300,
                                        zorder=2, marker='+', adjust_lims=True)
    lnB = lnBayes[i]
    lnB_str = 'NaN' if np.isnan(lnB[0]) else f'{lnB[0]:.3f}'r'$\pm$'f'{lnB[1]:.3f}'
    title_str = f'{eventname}: 'r'$\ln\mathcal B =$'+lnB_str
    fig = cp.corner_plots[0].fig
    fig.suptitle(title_str, y=0.97)
    fig.savefig(save_fn, bbox_inches='tight', pad_inches=0.2)
    plt.close(fig)

loaded posterior samples for GW12_1 run 0
ValueError: Too many bins for data range. Cannot create 46 finite-sized bins.. removing p_lensed and attempting to plot again...
loaded posterior samples for GW12_2 run 0
ValueError: Too many bins for data range. Cannot create 48 finite-sized bins.. removing p_lensed and attempting to plot again...
loaded posterior samples for GW12_3 run 0
loaded posterior samples for GW12_4 run 0
loaded posterior samples for GW12_5 run 0
loaded posterior samples for GW12_6 run 0
loaded posterior samples for GW12_7 run 0
loaded posterior samples for GW12_8 run 0
loaded posterior samples for GW12_9 run 0
loaded posterior samples for GW12_10 run 0
loaded posterior samples for GW12_11 run 0
loaded posterior samples for GW12_12 run 0
loaded posterior samples for GW12_13 run 0
ValueError: Too many bins for data range. Cannot create 45 finite-sized bins.. removing p_lensed and attempting to plot again...
loaded posterior samples for GW12_14 run 0
loaded posterior sam

/home/abbye.williams/cogwheel/cogwheel/plotting.py:523: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  ax.imshow(pdf, origin='lower', aspect='auto', extent=extent,
/home/abbye.williams/cogwheel/cogwheel/plotting.py:523: UserWarning: Attempting to set identical low and high xlims makes transformation singular; automatically expanding.
  ax.imshow(pdf, origin='lower', aspect='auto', extent=extent,


loaded posterior samples for GW12_47 run 0
ValueError: Too many bins for data range. Cannot create 72 finite-sized bins.. removing p_lensed and attempting to plot again...
exception! cannot convert float NaN to integer continuing to next event.
loaded posterior samples for GW12_48 run 0
loaded posterior samples for GW12_49 run 0
ValueError: Too many bins for data range. Cannot create 47 finite-sized bins.. removing p_lensed and attempting to plot again...
exception! Contour levels must be increasing continuing to next event.
loaded posterior samples for GW12_50 run 0
loaded posterior samples for GW12_51 run 0
loaded posterior samples for GW12_52 run 0
loaded posterior samples for GW12_53 run 0
loaded posterior samples for GW12_54 run 0
loaded posterior samples for GW12_55 run 0
loaded posterior samples for GW12_56 run 0
ValueError: Too many bins for data range. Cannot create 45 finite-sized bins.. removing p_lensed and attempting to plot again...
loaded posterior samples for GW12_57 ru

/home/abbye.williams/cogwheel/cogwheel/plotting.py:523: UserWarning: Attempting to set identical low and high ylims makes transformation singular; automatically expanding.
  ax.imshow(pdf, origin='lower', aspect='auto', extent=extent,


exception! Contour levels must be increasing continuing to next event.
loaded posterior samples for GW12_70 run 0
loaded posterior samples for GW12_72 run 0
loaded posterior samples for GW12_74 run 0
loaded posterior samples for GW12_75 run 0
loaded posterior samples for GW12_76 run 0
loaded posterior samples for GW12_77 run 0
loaded posterior samples for GW12_78 run 0
loaded posterior samples for GW12_79 run 0
loaded posterior samples for GW12_80 run 0
loaded posterior samples for GW12_81 run 0
ValueError: Too many bins for data range. Cannot create 47 finite-sized bins.. removing p_lensed and attempting to plot again...
loaded posterior samples for GW12_82 run 0
loaded posterior samples for GW12_83 run 0
loaded posterior samples for GW12_84 run 0
loaded posterior samples for GW12_85 run 0
loaded posterior samples for GW12_86 run 0
loaded posterior samples for GW12_87 run 0
loaded posterior samples for GW12_88 run 0
loaded posterior samples for GW12_89 run 0
ValueError: Too many bins 

In [14]:
# combine pdfs
pdf_paths = []
for i in idx_existing:
    fn = f'../data/plots/Halton_corner/GW{seed}_{i}_split.pdf'
    if os.path.exists(fn):
        pdf_paths.append(fn)
pdf_paths_str = ' '.join(pdf_paths)
bigpdf_path = f'../data/plots/Halton_corner/Halton_corner_split.pdf'
!gs -dBATCH -dNOPAUSE -q -sDEVICE=pdfwrite -dPDFSETTINGS=/prepress -sOutputFile={bigpdf_path} {pdf_paths_str}